In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd drive/MyDrive/

/content/drive/MyDrive


In [ ]:
train = pd.read_csv('traffic_V4.csv')
test = pd.read_csv('test_traffic_V4.csv')

print(f"학습 데이터 크기: {train.shape}")
print(f"테스트 데이터 크기: {test.shape}")

학습 데이터 크기: (249944, 94)
테스트 데이터 크기: (50000, 93)


In [ ]:
TARGET = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"피처 수: {len(feature_cols)}")

피처 수: 90


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(train)):
    print(f"── Fold {fold + 1} ──")
    X_tr = train.loc[tr_idx, feature_cols]
    y_tr = train.loc[tr_idx, TARGET]
    X_val = train.loc[val_idx, feature_cols]
    y_val = train.loc[val_idx, TARGET]

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=7,
        l2_leaf_reg=3,
        loss_function='MAE',
        eval_metric='MAE',
        random_seed=42,
        task_type='CPU',
        verbose=100
    )

    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        use_best_model=True
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(test[feature_cols]) / 5

── Fold 1 ──
0:	learn: 14.1949263	test: 14.2519939	best: 14.2519939 (0)	total: 177ms	remaining: 2m 56s
100:	learn: 9.3653734	test: 9.4537161	best: 9.4537161 (100)	total: 13.3s	remaining: 1m 58s
200:	learn: 9.2486971	test: 9.3753553	best: 9.3753553 (200)	total: 27.6s	remaining: 1m 49s
300:	learn: 9.1335325	test: 9.3059064	best: 9.3059064 (300)	total: 39.6s	remaining: 1m 31s
400:	learn: 9.0309761	test: 9.2457773	best: 9.2457773 (400)	total: 52.2s	remaining: 1m 17s
500:	learn: 8.9410588	test: 9.1916234	best: 9.1916234 (500)	total: 1m 5s	remaining: 1m 5s
600:	learn: 8.8649431	test: 9.1482249	best: 9.1482249 (600)	total: 1m 18s	remaining: 52.4s
700:	learn: 8.7928947	test: 9.1077778	best: 9.1077778 (700)	total: 1m 32s	remaining: 39.3s
800:	learn: 8.7284656	test: 9.0720058	best: 9.0720058 (800)	total: 1m 45s	remaining: 26.2s
900:	learn: 8.6676727	test: 9.0407816	best: 9.0407816 (900)	total: 1m 58s	remaining: 13s
999:	learn: 8.6104281	test: 9.0111287	best: 9.0111287 (999)	total: 2m 11s	remaini

In [ ]:
oof_mae = mean_absolute_error(train[TARGET], oof_preds)
print(f"OOF MAE: {oof_mae:.4f}")

OOF MAE: 8.9670


In [ ]:
submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
submission.to_csv('./submission_V18.csv', index=False)
print("submission.csv 저장 완료.")

submission.csv 저장 완료.
